# E-Commerce Customer Intelligence & Sales Analytics
## Notebook 02 — Data Cleaning & Preprocessing

**IBM SkillsBuild Data Analytics with AI Internship 2026**

---

### Scope

This notebook implements all approved data-cleaning decisions from the Notebook 01 audit.  
It produces clearly separated analytical datasets for downstream use.

**The original `data/online_retail_II.xlsx` is never modified.**

### Approved Cleaning Decisions (summary)

| # | Decision | Action |
|---|----------|--------|
| A | Exact duplicates | Remove from analytical datasets; report counts |
| B | Missing Customer ID | Retain for transaction analysis; exclude only for customer-level analysis |
| C | Zero / negative Price | Classify separately; exclude from merchandise sales |
| D | Non-standard StockCodes | Classify by type; exclude non-merchandise from product analysis |
| E | Negative Qty on non-C invoices | Classify into transaction types; do not auto-relabel |
| F | Multiple descriptions per StockCode | Build canonical description separately; preserve originals |

### Datasets produced

| Variable | Contents |
|----------|----------|
| `raw` | Full combined dataset, no rows removed, source sheet tagged |
| `cleaned_transactions` | Raw minus exact duplicates; all classification fields added |
| `valid_merchandise_transactions` | Positive-price, positive-quantity, merchandise-only rows |
| `customer_transactions` | `valid_merchandise_transactions` restricted to rows with a Customer ID |

---

> **Stop condition:** This notebook ends after cleaning and preprocessing.  
> RFM, clustering, cohort analysis, modelling, and dashboards are out of scope here.

## 1. Imports & Configuration

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ── Display settings ─────────────────────────────────────────────────────────
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 80)
pd.set_option('display.float_format', '{:,.4f}'.format)
pd.set_option('display.width', 130)

# Suppress openpyxl data-validation warnings (cosmetic only; no errors hidden)
warnings.filterwarnings(
    'ignore',
    message='.*data validation.*',
    category=UserWarning,
    module='openpyxl'
)

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_PATH   = '../data/online_retail_II.xlsx'
FIGURES_DIR = '../outputs/figures/'
os.makedirs(FIGURES_DIR, exist_ok=True)

# ── Plot style ─────────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 120, 'figure.facecolor': 'white'})

print('Imports OK')
print(f'pandas  {pd.__version__}  |  numpy  {np.__version__}')

## 2. Helper Functions

In [ ]:
def save_figure(filename: str) -> None:
    """Save the current matplotlib figure to the figures directory."""
    path = os.path.join(FIGURES_DIR, filename)
    plt.savefig(path, bbox_inches='tight')
    print(f'Figure saved → {path}')


def row_count_report(label: str, df: pd.DataFrame) -> None:
    """Print a one-line row-count summary."""
    print(f'  {label:<45}: {len(df):>10,} rows')


def reconciliation_table(counts: dict) -> pd.DataFrame:
    """
    Build a reconciliation DataFrame from an ordered dict of
    {label: count} pairs. Computes cumulative totals.
    """
    df = pd.DataFrame(list(counts.items()), columns=['Category', 'Row Count'])
    df['Row Count'] = df['Row Count'].apply(lambda x: f'{int(x):,}')
    return df


print('Helper functions defined.')

## 3. Load Raw Data

Reload both sheets from the untouched Excel file and combine them exactly as in Notebook 01.

In [ ]:
xl_file     = pd.ExcelFile(DATA_PATH, engine='openpyxl')
sheet_names = xl_file.sheet_names
print(f'Sheets: {sheet_names}')

sheets = {}
for name in sheet_names:
    print(f'  Loading "{name}" ...', end=' ')
    df = pd.read_excel(
        DATA_PATH,
        sheet_name=name,
        engine='openpyxl',
        dtype={'Invoice': str, 'StockCode': str},
    )
    sheets[name] = df
    print(f'{len(df):,} rows')

tagged = []
for name, df in sheets.items():
    tmp = df.copy()
    tmp['_sheet'] = name
    tagged.append(tmp)

raw = pd.concat(tagged, ignore_index=True)
raw['InvoiceDate'] = pd.to_datetime(raw['InvoiceDate'], errors='coerce')

RAW_ROW_COUNT = len(raw)

print(f'\nRaw combined rows: {RAW_ROW_COUNT:,}')
assert RAW_ROW_COUNT == sum(len(d) for d in sheets.values()), \
    'Row count mismatch after concatenation'
print('Assertion passed: combined row count == sum of sheets.')

In [ ]:
# Confirm original columns are intact
EXPECTED_COLS = ['Invoice', 'StockCode', 'Description', 'Quantity',
                 'InvoiceDate', 'Price', 'Customer ID', 'Country']
for col in EXPECTED_COLS:
    assert col in raw.columns, f'Missing expected column: {col}'
print('All expected columns present in raw dataset.')
print(f'Columns: {list(raw.columns)}')

## 4. Decision A — Remove Exact Duplicate Rows

**Rule:** Remove rows where every column value is identical to another row.  
**Scope:** Only exact, full-row duplicates are removed — not near-duplicates or repeated invoice/product combinations.  
**Validation:** Confirm that (retained + removed) = raw total.

In [ ]:
# Identify exact duplicates (all columns including _sheet)
dup_mask       = raw.duplicated(keep='first')
n_duplicates   = dup_mask.sum()
n_after_dedup  = RAW_ROW_COUNT - n_duplicates
pct_removed    = n_duplicates / RAW_ROW_COUNT * 100

print('Decision A — Exact Duplicate Removal')
print(f'  Original row count    : {RAW_ROW_COUNT:,}')
print(f'  Exact duplicates found: {n_duplicates:,}')
print(f'  Rows after dedup      : {n_after_dedup:,}')
print(f'  Percentage removed    : {pct_removed:.3f}%')

In [ ]:
# Show a sample of duplicated rows to confirm they are genuine exact copies
dup_sample = raw[raw.duplicated(keep=False)].sort_values(
    ['Invoice', 'StockCode', 'Quantity', 'Price']
).head(10)
print('Sample exact duplicates (first 10 of duplicated-flag rows):')
display(dup_sample[['Invoice', 'StockCode', 'Description', 'Quantity',
                    'InvoiceDate', 'Price', 'Customer ID', 'Country', '_sheet']])

In [ ]:
# Remove exact duplicates — keep first occurrence
cleaned_transactions = raw[~dup_mask].copy()
cleaned_transactions = cleaned_transactions.reset_index(drop=True)

# Validation assertions
assert len(cleaned_transactions) == n_after_dedup, \
    'Row count mismatch after deduplication'
assert len(cleaned_transactions) + n_duplicates == RAW_ROW_COUNT, \
    'Retained + removed != original'
assert cleaned_transactions.duplicated().sum() == 0, \
    'Remaining duplicates found after deduplication'

print('Assertions passed:')
print(f'  Retained ({len(cleaned_transactions):,}) + removed ({n_duplicates:,}) == raw ({RAW_ROW_COUNT:,})')
print(f'  Zero exact duplicates remaining in cleaned_transactions.')

## 5. Decision B — Missing Customer ID

**Rule:** Do not remove rows with missing Customer ID globally.  
- All rows are retained in `cleaned_transactions`.  
- Customer-level analyses must filter to `Customer ID` present rows.

In [ ]:
n_total_cleaned   = len(cleaned_transactions)
n_has_cid         = cleaned_transactions['Customer ID'].notna().sum()
n_missing_cid     = cleaned_transactions['Customer ID'].isna().sum()
pct_missing_cid   = n_missing_cid / n_total_cleaned * 100
n_unique_cust     = cleaned_transactions['Customer ID'].nunique()

print('Decision B — Missing Customer ID')
print(f'  Rows with Customer ID     : {n_has_cid:,}')
print(f'  Rows without Customer ID  : {n_missing_cid:,}')
print(f'  Missing percentage        : {pct_missing_cid:.2f}%')
print(f'  Unique identified customers: {n_unique_cust:,}')
print()
print('Handling: rows without Customer ID are RETAINED in cleaned_transactions.')
print('          Customer-level analyses must use: df[df["Customer ID"].notna()]')

## 6. Decision C — Price Classification (Zero and Negative)

**Rule:** `Price <= 0` rows are classified separately.  
- Zero-price rows: non-standard records (samples, internal transfers, missing pricing data).  
- Negative-price rows: accounting bad-debt adjustments.  
Both are preserved in `cleaned_transactions` but flagged and excluded from `valid_merchandise_transactions`.

In [ ]:
# Add price classification flags
cleaned_transactions['is_zero_price']     = cleaned_transactions['Price'] == 0
cleaned_transactions['is_negative_price'] = cleaned_transactions['Price'] < 0

n_zero_price     = cleaned_transactions['is_zero_price'].sum()
n_neg_price      = cleaned_transactions['is_negative_price'].sum()
n_positive_price = (cleaned_transactions['Price'] > 0).sum()

print('Decision C — Price Classification')
print(f'  Positive Price (> 0) rows : {n_positive_price:,}')
print(f'  Zero Price (= 0) rows     : {n_zero_price:,}')
print(f'  Negative Price (< 0) rows : {n_neg_price:,}')
print()

# Show the negative-price records in full (expected ~5)
if n_neg_price > 0:
    print('Negative-price records (bad-debt adjustments):')
    neg_price_rows = cleaned_transactions[cleaned_transactions['is_negative_price']]
    display(neg_price_rows[['Invoice', 'StockCode', 'Description', 'Quantity',
                             'Price', 'Customer ID', 'Country', '_sheet']])

# Show top zero-price StockCodes
if n_zero_price > 0:
    print('\nTop 20 StockCodes with zero Price:')
    display(
        cleaned_transactions[cleaned_transactions['is_zero_price']]
        .groupby(['StockCode', 'Description'])
        .size()
        .reset_index(name='zero_price_rows')
        .sort_values('zero_price_rows', ascending=False)
        .head(20)
    )

## 7. Decision D — Non-Standard StockCode Classification

**Rule:** Build a `product_type` classification field.  
Categories: `MERCHANDISE`, `POSTAGE`, `FEE_OR_CHARGE`, `DISCOUNT`, `MANUAL_ADJUSTMENT`, `SAMPLE`, `TEST`, `GIFT_VOUCHER`, `BAD_DEBT`, `UNCLASSIFIED_NON_STANDARD`.

Rules are applied in order of specificity. Standard 5-digit codes are classified as `MERCHANDISE` by default.  
Non-standard codes that match no rule fall to `UNCLASSIFIED_NON_STANDARD`.

In [ ]:
import re

def classify_stockcode(code: str, description: str) -> str:
    """
    Classify a StockCode into a product_type category.
    Rules are applied in priority order (most specific first).
    Returns one of the defined category strings.
    """
    code = str(code).strip().upper() if pd.notna(code) else ''
    desc = str(description).strip().upper() if pd.notna(description) else ''

    # ── 1. Test records ───────────────────────────────────────────────────────
    if re.match(r'^TEST', code) or 'TEST' in desc:
        return 'TEST'

    # ── 2. Bad debt adjustments ───────────────────────────────────────────────
    if code in ('B', 'ADJUST2') or 'BAD DEBT' in desc or 'BAD-DEBT' in desc:
        return 'BAD_DEBT'

    # ── 3. Gift vouchers ──────────────────────────────────────────────────────
    if code.startswith('GIFT') or 'GIFT VOUCHER' in desc or 'GIFT_0' in code:
        return 'GIFT_VOUCHER'

    # ── 4. Postage / shipping ─────────────────────────────────────────────────
    if code in ('POST', 'DOT', 'C2') or 'POSTAGE' in desc or 'CARRIAGE' in desc:
        return 'POSTAGE'

    # ── 5. Fees and external charges ─────────────────────────────────────────
    if code in ('BANK CHARGES', 'BANKCHARGES', 'AMAZONFEE') \
            or 'BANK CHARGE' in desc or 'AMAZON FEE' in desc or 'FEE' in desc:
        return 'FEE_OR_CHARGE'

    # ── 6. Discounts ─────────────────────────────────────────────────────────
    if code == 'D' or 'DISCOUNT' in desc:
        return 'DISCOUNT'

    # ── 7. Samples ────────────────────────────────────────────────────────────
    if code == 'S' or 'SAMPLE' in desc:
        return 'SAMPLE'

    # ── 8. Manual adjustments ────────────────────────────────────────────────
    if code in ('M', 'ADJUST', 'CRUK') \
            or 'MANUAL' in desc or 'ADJUST' in desc or 'CRUK' in desc:
        return 'MANUAL_ADJUSTMENT'

    # ── 9. Standard merchandise: 5-digit numeric (with optional trailing letter)
    if re.match(r'^\d{5}[A-Z]?$', code):
        return 'MERCHANDISE'

    # ── 10. Anything else that is non-standard ────────────────────────────────
    return 'UNCLASSIFIED_NON_STANDARD'


# Apply classification
cleaned_transactions['product_type'] = cleaned_transactions.apply(
    lambda row: classify_stockcode(row['StockCode'], row['Description']),
    axis=1
)

print('product_type classification applied.')
pt_counts = cleaned_transactions['product_type'].value_counts()
display(pt_counts.rename('row_count').reset_index().rename(columns={'index': 'product_type'}))

In [ ]:
# Inspect UNCLASSIFIED_NON_STANDARD codes to confirm they are truly miscellaneous
unclassified = (
    cleaned_transactions[cleaned_transactions['product_type'] == 'UNCLASSIFIED_NON_STANDARD']
    .groupby(['StockCode', 'Description'])
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
)
print(f'UNCLASSIFIED_NON_STANDARD unique StockCodes: {unclassified["StockCode"].nunique():,}')
print('Top 30 by row count:')
display(unclassified.head(30))

In [ ]:
# Summary: merchandise vs non-merchandise
merch_count    = (cleaned_transactions['product_type'] == 'MERCHANDISE').sum()
non_merch_count = (cleaned_transactions['product_type'] != 'MERCHANDISE').sum()
pct_merch      = merch_count / len(cleaned_transactions) * 100

print(f'MERCHANDISE rows      : {merch_count:,} ({pct_merch:.2f}%)')
print(f'Non-MERCHANDISE rows  : {non_merch_count:,} ({100 - pct_merch:.2f}%)')

## 8. Decision E — Classify Negative Quantity on Non-Cancelled Invoices

**Rule:** First identify cancellation invoices (starting with `C`).  
Then assign a `transaction_type` to every row using the following logic:

| Condition | transaction_type |
|-----------|------------------|
| Invoice starts with `C` | `CANCELLED_INVOICE` |
| Non-C invoice, Qty < 0 | `RETURN_OR_NEGATIVE_ADJUSTMENT` |
| Non-C invoice, Qty = 0 | `ZERO_QUANTITY` |
| Non-C invoice, Qty > 0, Price > 0 | `SALE` |
| Non-C invoice, Qty > 0, Price = 0 | `ZERO_PRICE_POSITIVE_QTY` |
| Non-C invoice, Qty > 0, Price < 0 | `NEGATIVE_PRICE_POSITIVE_QTY` |
| Fallthrough | `OTHER_ADJUSTMENT` |

In [ ]:
def assign_transaction_type(row) -> str:
    """
    Assign a transaction_type label based on Invoice prefix, Quantity, and Price.
    Rules applied in priority order.
    """
    invoice = str(row['Invoice']).strip().upper()
    qty     = row['Quantity']
    price   = row['Price']

    # Cancellation invoices (Invoice starts with C)
    if invoice.startswith('C'):
        return 'CANCELLED_INVOICE'

    # Non-cancellation invoice — negative quantity
    if qty < 0:
        return 'RETURN_OR_NEGATIVE_ADJUSTMENT'

    # Non-cancellation invoice — zero quantity
    if qty == 0:
        return 'ZERO_QUANTITY'

    # Non-cancellation, positive quantity
    if qty > 0:
        if price > 0:
            return 'SALE'
        if price == 0:
            return 'ZERO_PRICE_POSITIVE_QTY'
        if price < 0:
            return 'NEGATIVE_PRICE_POSITIVE_QTY'

    return 'OTHER_ADJUSTMENT'


cleaned_transactions['transaction_type'] = cleaned_transactions.apply(
    assign_transaction_type, axis=1
)

print('transaction_type classification applied.')
tt_counts = cleaned_transactions['transaction_type'].value_counts()
display(tt_counts.rename('row_count').reset_index().rename(columns={'index': 'transaction_type'}))

In [ ]:
# Inspect the RETURN_OR_NEGATIVE_ADJUSTMENT rows
returns_non_cancel = cleaned_transactions[
    cleaned_transactions['transaction_type'] == 'RETURN_OR_NEGATIVE_ADJUSTMENT'
]
n_returns_non_cancel = len(returns_non_cancel)

print(f'RETURN_OR_NEGATIVE_ADJUSTMENT rows (non-C invoices, Qty < 0): {n_returns_non_cancel:,}')
print('\nSample (first 10):')
display(returns_non_cancel[
    ['Invoice', 'StockCode', 'Description', 'Quantity', 'Price', 'Customer ID', 'Country']
].head(10))

print('\nPrice distribution for these rows:')
print(returns_non_cancel['Price'].describe())

## 9. Decision F — Canonical Description

**Rule:** Do not overwrite the original `Description` column.  
Build a separate `canonical_description` field per StockCode using the most frequently observed non-null description.

In [ ]:
# Build canonical description: modal (most frequent) non-null Description per StockCode
def modal_description(series: pd.Series) -> str:
    """Return the most frequent non-null value in a Series, or None if all null."""
    non_null = series.dropna()
    if len(non_null) == 0:
        return None
    return non_null.mode().iloc[0]


canonical_desc_map = (
    cleaned_transactions
    .groupby('StockCode')['Description']
    .agg(modal_description)
)

# Attach to cleaned_transactions
cleaned_transactions['canonical_description'] = (
    cleaned_transactions['StockCode'].map(canonical_desc_map)
)

# How many StockCodes required canonicalization (had >1 unique description)?
desc_per_stock = (
    cleaned_transactions.dropna(subset=['Description'])
    .groupby('StockCode')['Description']
    .nunique()
)
n_multi_desc = (desc_per_stock > 1).sum()
n_single_desc = (desc_per_stock == 1).sum()

print(f'StockCodes with single unique description    : {n_single_desc:,}')
print(f'StockCodes requiring canonicalization (>1)   : {n_multi_desc:,}')
print(f'Rule: most-frequent non-null Description per StockCode')
print(f'Original Description column: preserved, not modified.')
print(f'canonical_description column: added alongside.')

In [ ]:
# Show examples where canonical differs from transaction-level description
mismatch = cleaned_transactions[
    cleaned_transactions['Description'].notna() &
    (cleaned_transactions['Description'] != cleaned_transactions['canonical_description'])
][['StockCode', 'Description', 'canonical_description']].drop_duplicates().head(15)

print('Sample rows where Description != canonical_description:')
display(mismatch)

## 10. Additional Feature Engineering

Add the remaining boolean/indicator fields and the revenue field.

In [ ]:
# ── Boolean indicator fields ────────────────────────────────────────────────
cleaned_transactions['is_cancelled'] = (
    cleaned_transactions['transaction_type'] == 'CANCELLED_INVOICE'
)
cleaned_transactions['is_negative_quantity'] = (
    cleaned_transactions['Quantity'] < 0
)
cleaned_transactions['is_positive_sale'] = (
    cleaned_transactions['transaction_type'] == 'SALE'
)

# ── Revenue ─────────────────────────────────────────────────────────────────
# Revenue = Quantity * Price on ALL rows (can be negative for cancellations/returns)
cleaned_transactions['revenue'] = (
    cleaned_transactions['Quantity'] * cleaned_transactions['Price']
)

# Assertion: revenue formula is exact
recalc = cleaned_transactions['Quantity'] * cleaned_transactions['Price']
assert (recalc.round(8) == cleaned_transactions['revenue'].round(8)).all(), \
    'ASSERTION FAILED: revenue != Quantity * Price'

print('Feature fields added:')
feature_cols = [
    'is_cancelled', 'is_negative_quantity', 'is_positive_sale',
    'is_zero_price', 'is_negative_price',
    'transaction_type', 'product_type', 'canonical_description', 'revenue'
]
for col in feature_cols:
    print(f'  {col:<30}: {cleaned_transactions[col].dtype}')

print('\nAssertion passed: revenue == Quantity * Price on all rows.')

In [ ]:
# Summary of boolean fields
bool_summary = pd.DataFrame({
    'Field': ['is_cancelled', 'is_negative_quantity', 'is_positive_sale',
              'is_zero_price', 'is_negative_price'],
    'True Count': [
        cleaned_transactions['is_cancelled'].sum(),
        cleaned_transactions['is_negative_quantity'].sum(),
        cleaned_transactions['is_positive_sale'].sum(),
        cleaned_transactions['is_zero_price'].sum(),
        cleaned_transactions['is_negative_price'].sum(),
    ]
})
bool_summary['True %'] = (bool_summary['True Count'] / len(cleaned_transactions) * 100).round(2)
display(bool_summary)

## 11. Create Separated Analytical Datasets

Four datasets are produced. No full copies of the entire cleaned dataset are made unnecessarily — views/filters are used where possible.

| Dataset | Filter applied |
|---------|---------------|
| `cleaned_transactions` | Raw minus exact duplicates; all classification fields added |
| `valid_merchandise_transactions` | `transaction_type == SALE` AND `product_type == MERCHANDISE` |
| `customer_transactions` | `valid_merchandise_transactions` with `Customer ID` present |
| `merchandise_product_transactions` | `product_type == MERCHANDISE` (includes cancellations for net revenue calc) |

In [ ]:
# ── valid_merchandise_transactions ──────────────────────────────────────────
# Positive-price, positive-quantity, merchandise StockCodes only
valid_merchandise_transactions = cleaned_transactions[
    (cleaned_transactions['transaction_type'] == 'SALE') &
    (cleaned_transactions['product_type'] == 'MERCHANDISE')
].copy()

# ── customer_transactions ────────────────────────────────────────────────────
# Valid merchandise transactions restricted to identified customers
customer_transactions = valid_merchandise_transactions[
    valid_merchandise_transactions['Customer ID'].notna()
].copy()

# ── merchandise_product_transactions ────────────────────────────────────────
# All merchandise rows including cancellations (for net revenue per product)
merchandise_product_transactions = cleaned_transactions[
    cleaned_transactions['product_type'] == 'MERCHANDISE'
].copy()

print('Analytical datasets created:')
row_count_report('cleaned_transactions', cleaned_transactions)
row_count_report('valid_merchandise_transactions', valid_merchandise_transactions)
row_count_report('customer_transactions', customer_transactions)
row_count_report('merchandise_product_transactions', merchandise_product_transactions)

In [ ]:
# Validate no unexpected rows slipped through

# valid_merchandise_transactions: all must be SALE + MERCHANDISE
assert (valid_merchandise_transactions['transaction_type'] == 'SALE').all(), \
    'valid_merchandise_transactions contains non-SALE rows'
assert (valid_merchandise_transactions['product_type'] == 'MERCHANDISE').all(), \
    'valid_merchandise_transactions contains non-MERCHANDISE rows'
assert (valid_merchandise_transactions['Quantity'] > 0).all(), \
    'valid_merchandise_transactions contains non-positive Quantity'
assert (valid_merchandise_transactions['Price'] > 0).all(), \
    'valid_merchandise_transactions contains non-positive Price'

# customer_transactions: all must have Customer ID
assert customer_transactions['Customer ID'].notna().all(), \
    'customer_transactions contains rows with missing Customer ID'

print('Dataset validation assertions all passed.')

## 12. Reconciliation Tables

Full row-count reconciliation to confirm no rows were silently lost or double-counted.

In [ ]:
# ── Reconciliation 1: Raw → Cleaned ─────────────────────────────────────────
print('=== Reconciliation 1: Raw → Cleaned Transactions ===')
print(f'  Raw rows                      : {RAW_ROW_COUNT:>10,}')
print(f'  Exact duplicates removed      : {n_duplicates:>10,}')
print(f'  Cleaned transactions          : {len(cleaned_transactions):>10,}')
assert len(cleaned_transactions) == RAW_ROW_COUNT - n_duplicates
print('  [OK] Raw - duplicates == cleaned_transactions')

In [ ]:
# ── Reconciliation 2: Transaction types within cleaned_transactions ──────────
print('=== Reconciliation 2: Transaction Type Breakdown ===')
tt_recon = cleaned_transactions['transaction_type'].value_counts().sort_values(ascending=False)
tt_total = tt_recon.sum()
print(f'  Sum of all transaction_type counts : {tt_total:,}')
print(f'  cleaned_transactions rows          : {len(cleaned_transactions):,}')
assert tt_total == len(cleaned_transactions), \
    'transaction_type counts do not sum to cleaned_transactions'
print('  [OK] Counts reconcile.\n')

for label, count in tt_recon.items():
    pct = count / len(cleaned_transactions) * 100
    print(f'  {label:<40}: {count:>10,}  ({pct:.2f}%)')

In [ ]:
# ── Reconciliation 3: Product type breakdown ─────────────────────────────────
print('=== Reconciliation 3: Product Type Breakdown ===')
pt_recon = cleaned_transactions['product_type'].value_counts().sort_values(ascending=False)
pt_total = pt_recon.sum()
assert pt_total == len(cleaned_transactions), \
    'product_type counts do not sum to cleaned_transactions'

for label, count in pt_recon.items():
    pct = count / len(cleaned_transactions) * 100
    print(f'  {label:<40}: {count:>10,}  ({pct:.2f}%)')
print(f'  {"TOTAL":<40}: {pt_total:>10,}')
print('  [OK] Counts reconcile.')

In [ ]:
# ── Reconciliation 4: Path from cleaned → valid_merchandise_transactions ────
print('=== Reconciliation 4: Exclusions from cleaned → valid_merchandise_transactions ===')

n_cleaned       = len(cleaned_transactions)
n_non_sale      = (cleaned_transactions['transaction_type'] != 'SALE').sum()
n_non_merch     = (
    (cleaned_transactions['transaction_type'] == 'SALE') &
    (cleaned_transactions['product_type'] != 'MERCHANDISE')
).sum()
n_valid_merch   = len(valid_merchandise_transactions)

print(f'  cleaned_transactions                    : {n_cleaned:>10,}')
print(f'  Excluded (non-SALE transaction_type)    : {n_non_sale:>10,}')
print(f'  Excluded (SALE but non-MERCHANDISE)     : {n_non_merch:>10,}')
print(f'  valid_merchandise_transactions          : {n_valid_merch:>10,}')

assert n_cleaned - n_non_sale - n_non_merch == n_valid_merch, \
    'Reconciliation 4 mismatch'
print('  [OK] Counts reconcile.')

In [ ]:
# ── Reconciliation 5: customer_transactions ──────────────────────────────────
print('=== Reconciliation 5: customer_transactions ===')

n_vmt           = len(valid_merchandise_transactions)
n_cust_missing  = valid_merchandise_transactions['Customer ID'].isna().sum()
n_cust_present  = len(customer_transactions)

print(f'  valid_merchandise_transactions          : {n_vmt:>10,}')
print(f'  Rows without Customer ID (excluded)     : {n_cust_missing:>10,}')
print(f'  customer_transactions                   : {n_cust_present:>10,}')

assert n_vmt - n_cust_missing == n_cust_present, \
    'Reconciliation 5 mismatch'
print('  [OK] Counts reconcile.')

## 13. Cleaning Visualizations

Three charts to illustrate the cleaning outcome.

### 13.1 Row Count Waterfall: Raw → Datasets

In [ ]:
waterfall_labels = [
    'Raw (all sheets)',
    'After dedup',
    'SALE rows',
    'SALE + MERCHANDISE',
    'SALE + MERCH + Customer ID',
]
waterfall_values = [
    RAW_ROW_COUNT,
    len(cleaned_transactions),
    (cleaned_transactions['transaction_type'] == 'SALE').sum(),
    len(valid_merchandise_transactions),
    len(customer_transactions),
]

colors = [sns.color_palette('muted')[i] for i in [0, 2, 4, 1, 3]]

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar(waterfall_labels, waterfall_values, color=colors, edgecolor='white', linewidth=0.8)
for bar, val in zip(bars, waterfall_values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max(waterfall_values) * 0.01,
        f'{val:,}',
        ha='center', va='bottom', fontsize=9.5
    )
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.set_title('Row Count at Each Cleaning Stage', fontsize=13, fontweight='bold')
ax.set_ylabel('Number of Rows')
ax.set_xlabel('Dataset Stage')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
save_figure('06_cleaning_waterfall.png')
plt.show()

### 13.2 Transaction Type Distribution (Cleaned Transactions)

In [ ]:
tt_plot = cleaned_transactions['transaction_type'].value_counts().sort_values()

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(tt_plot.index, tt_plot.values,
        color=sns.color_palette('muted', n_colors=len(tt_plot)), edgecolor='white')
for i, (idx, val) in enumerate(zip(tt_plot.index, tt_plot.values)):
    ax.text(val + max(tt_plot.values) * 0.005, i, f'{val:,}', va='center', fontsize=9)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.set_title('Transaction Type Distribution (Cleaned Transactions)', fontsize=13, fontweight='bold')
ax.set_xlabel('Number of Rows')
ax.set_ylabel('Transaction Type')
plt.tight_layout()
save_figure('07_transaction_type_distribution.png')
plt.show()

### 13.3 Product Type Distribution

In [ ]:
pt_plot = cleaned_transactions['product_type'].value_counts().sort_values()

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(pt_plot.index, pt_plot.values,
        color=sns.color_palette('muted', n_colors=len(pt_plot)), edgecolor='white')
for i, (idx, val) in enumerate(zip(pt_plot.index, pt_plot.values)):
    ax.text(val + max(pt_plot.values) * 0.005, i, f'{val:,}', va='center', fontsize=9)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.set_title('Product Type Distribution (Cleaned Transactions)', fontsize=13, fontweight='bold')
ax.set_xlabel('Number of Rows')
ax.set_ylabel('Product Type')
plt.tight_layout()
save_figure('08_product_type_distribution.png')
plt.show()

## 14. Final Validation Summary

In [ ]:
print('=== Final Validation Suite ===')

checks = []

def check(label: str, condition: bool, detail: str = '') -> None:
    status = 'PASS' if condition else 'FAIL'
    checks.append((label, status, detail))
    print(f'  [{status}] {label}' + (f' — {detail}' if detail else ''))
    if not condition:
        raise AssertionError(f'Validation failed: {label}')

# Row counts
check('Raw - duplicates == cleaned_transactions',
      RAW_ROW_COUNT - n_duplicates == len(cleaned_transactions))

# No exact duplicates remain
check('Zero duplicates in cleaned_transactions',
      cleaned_transactions.duplicated().sum() == 0)

# Transaction type completeness
check('Every row has a transaction_type',
      cleaned_transactions['transaction_type'].notna().all())

# Product type completeness
check('Every row has a product_type',
      cleaned_transactions['product_type'].notna().all())

# Revenue formula
rev_check = (cleaned_transactions['Quantity'] * cleaned_transactions['Price']).round(8)
check('revenue == Quantity * Price everywhere',
      (rev_check == cleaned_transactions['revenue'].round(8)).all())

# valid_merchandise_transactions: positive qty and price
check('valid_merchandise_transactions: Quantity > 0',
      (valid_merchandise_transactions['Quantity'] > 0).all())
check('valid_merchandise_transactions: Price > 0',
      (valid_merchandise_transactions['Price'] > 0).all())

# customer_transactions: no missing Customer ID
check('customer_transactions: no missing Customer ID',
      customer_transactions['Customer ID'].notna().all())

# is_cancelled consistency with transaction_type
cancel_flag_count  = cleaned_transactions['is_cancelled'].sum()
cancel_type_count  = (cleaned_transactions['transaction_type'] == 'CANCELLED_INVOICE').sum()
check('is_cancelled consistent with transaction_type',
      cancel_flag_count == cancel_type_count,
      f'{cancel_flag_count:,} vs {cancel_type_count:,}')

# is_positive_sale consistency
pos_flag_count = cleaned_transactions['is_positive_sale'].sum()
pos_type_count = (cleaned_transactions['transaction_type'] == 'SALE').sum()
check('is_positive_sale consistent with transaction_type',
      pos_flag_count == pos_type_count,
      f'{pos_flag_count:,} vs {pos_type_count:,}')

print(f'\nAll {len(checks)} validation checks passed.')

## 15. Cleaning Summary

All values below are computed from the actual data — none are fabricated.

In [ ]:
# ── Revenue summary on valid_merchandise_transactions ───────────────────────
total_sales_revenue  = valid_merchandise_transactions['revenue'].sum()
cancel_revenue       = cleaned_transactions.loc[
    cleaned_transactions['is_cancelled'], 'revenue'
].sum()
returns_revenue      = cleaned_transactions.loc[
    cleaned_transactions['transaction_type'] == 'RETURN_OR_NEGATIVE_ADJUSTMENT',
    'revenue'
].sum()
net_revenue_all      = cleaned_transactions['revenue'].sum()

print('=== Cleaning Summary ===')
print()
print('--- A. Row Counts ---')
print(f'  Raw rows (both sheets combined)         : {RAW_ROW_COUNT:>12,}')
print(f'  Exact duplicates removed                : {n_duplicates:>12,}  ({pct_removed:.3f}%)')
print(f'  Cleaned transactions (deduplicated)     : {len(cleaned_transactions):>12,}')
print()
print('--- B. Transaction Type Breakdown ---')
for label, count in cleaned_transactions['transaction_type'].value_counts().items():
    pct = count / len(cleaned_transactions) * 100
    print(f'  {label:<42}: {count:>10,}  ({pct:.2f}%)')
print()
print('--- C. Product Type Breakdown ---')
for label, count in cleaned_transactions['product_type'].value_counts().items():
    pct = count / len(cleaned_transactions) * 100
    print(f'  {label:<42}: {count:>10,}  ({pct:.2f}%)')
print()
print('--- D. Price Anomalies ---')
print(f'  Zero-price rows                         : {n_zero_price:>12,}')
print(f'  Negative-price rows (bad-debt adj.)     : {n_neg_price:>12,}')
print()
print('--- E. Customer ID Coverage ---')
print(f'  Rows with Customer ID                   : {n_has_cid:>12,}  ({100-pct_missing_cid:.2f}%)')
print(f'  Rows without Customer ID                : {n_missing_cid:>12,}  ({pct_missing_cid:.2f}%)')
print(f'  Unique identified customers             : {n_unique_cust:>12,}')
print()
print('--- F. Description Canonicalization ---')
print(f'  StockCodes with single description      : {n_single_desc:>12,}')
print(f'  StockCodes requiring canonicalization   : {n_multi_desc:>12,}')
print()
print('--- G. Analytical Datasets Produced ---')
row_count_report('cleaned_transactions', cleaned_transactions)
row_count_report('valid_merchandise_transactions', valid_merchandise_transactions)
row_count_report('customer_transactions', customer_transactions)
row_count_report('merchandise_product_transactions', merchandise_product_transactions)
print()
print('--- H. Revenue Summary ---')
print(f'  Gross revenue (valid_merchandise_txns)  : £{total_sales_revenue:>15,.2f}')
print(f'  Revenue on cancelled invoices           : £{cancel_revenue:>15,.2f}')
print(f'  Revenue on non-C negative-qty rows      : £{returns_revenue:>15,.2f}')
print(f'  Net revenue (all cleaned rows)          : £{net_revenue_all:>15,.2f}')

## 16. Assumptions and Unresolved Cases

| # | Item | Assumption / Status |
|---|------|---------------------|
| 1 | Exact duplicates | Treated as data-entry errors; first occurrence kept. If any represent legitimate repeated transactions (same customer, same product, same second), a timestamp-level review could refine this — not possible without sub-second precision in InvoiceDate. |
| 2 | `classify_stockcode` rule order | Rules are applied first-match in priority order. Any new non-standard code not matching a rule falls to `UNCLASSIFIED_NON_STANDARD` and should be reviewed in future data refreshes. |
| 3 | `RETURN_OR_NEGATIVE_ADJUSTMENT` | Classified separately from `CANCELLED_INVOICE`. These rows are preserved with their original Invoice. No assumption is made about whether they represent missed cancellation prefixes or genuine post-sale returns. |
| 4 | Negative-price rows | Classified as `BAD_DEBT` via StockCode `B` and description pattern. If additional bad-debt codes exist that are not in the training data, they will fall to `UNCLASSIFIED_NON_STANDARD`. |
| 5 | Currency | All monetary values assumed to be in GBP (British Pounds) as per dataset provenance. No currency conversion is applied. |
| 6 | `canonical_description` | Uses the modal (most frequent) non-null description per StockCode. Ties are broken by Pandas `mode().iloc[0]` (lexicographic first among ties). |

---

*End of Notebook 02 — Data Cleaning & Preprocessing.*  
*Proceed to Notebook 03 (Feature Engineering / EDA) only after reviewing the cleaning summary above.*